# Python Encapsulation

> 📘 **Python Mastery** · Module 08 — Object-Oriented Programming (OOP) · Lesson 5/5

Encapsulation means an object guards its own internals: outside code interacts through a small, controlled surface while the object enforces its own rules. This lesson finishes our banking example properly — by the end, a balance simply *cannot* go negative.

## 🎯 Learning Objectives

- Explain what an invariant is and why unprotected state breaks it
- Apply Python's underscore conventions (`_name`, `__name`) and know what each signals
- Explain name mangling and demonstrate that `__` attributes remain reachable
- Build validated getters/setters with `@property` that raise on bad input
- Create read-only and computed properties
- Decide when encapsulation genuinely matters

## 1. Why Protect Internal State?

An **invariant** is a rule that must always hold about an object — *"a balance is a number, and it is never negative."* If every attribute is wide open, any stray line of code can break the invariant silently, and soon every function in the program must defend itself against corrupt objects.

**Syntax:**

```python
class Naive:
    def __init__(self):
        self.balance = 0      # anyone can write ANYTHING here
```

**Example:** no guard rails, no complaints.

In [1]:
class NaiveAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance


acc = NaiveAccount("Sarah", 5000)

acc.balance = -999_999           # nothing stops nonsense
acc.balance = "one million"      # not even wrong TYPES are caught

print(acc.balance)               # the object happily carries corrupt state
# Invariant broken twice, zero errors raised. Exhausting to defend against.

one million


## 2. Python's Philosophy: "We're All Consenting Adults"

Unlike Java or C++, Python has no `private` keyword. The language trusts you: everything is **public by default**, and privacy is expressed through *conventions* that signal intent. You can always reach in — the conventions exist so readers know they shouldn't.

**Example:** reaching straight into an object is legal — and sometimes fine.

In [2]:
class Playlist:
    """Everything public; Python trusts you."""

    def __init__(self, title, songs):
        self.title = title
        self.songs = songs


night_drive = Playlist("Night Drive", ["Monpura", "Tumi Jake Bhalobasho"])

night_drive.songs.append("Ekhon Onek Raat")    # straight into the object's guts
print(night_drive.songs)
# Legal and sometimes perfectly fine -- Python gives you rope,
# plus conventions (next sections) to mark the "hands off" parts.

['Monpura', 'Tumi Jake Bhalobasho', 'Ekhon Onek Raat']


## 3. `_single_underscore`: Handle with Care

A leading underscore is a **convention**, not enforcement: `_name` means *"internal — part of the implementation, not the API."* Tooling respects the hint (`from module import *` skips underscore names, most IDEs gray them out), but nothing stops you.

**Syntax:**

```python
self.name     # public: documented, stable, safe to use
self._name    # internal: may change without notice; you were warned
```

**Example:**

In [3]:
class Report:
    def __init__(self, title, rows):
        self.title = title               # public: part of the API
        self.rows = rows                 # public
        self._row_count = len(rows)      # internal: derived detail
        self._cache = {}                 # internal: machinery

    def refresh(self):
        self._row_count = len(self.rows)
        return self._row_count


r = Report("Q3 Sales", [["a", 1], ["b", 2]])

print(r.title)          # encouraged
print(r._row_count)     # POSSIBLE -- but you were warned, colleague

Q3 Sales
2


## 4. `__double_underscore`: Name Mangling

A **double** leading underscore does something real: Python renames the attribute to `_ClassName__attr` inside the class where it was defined. Direct access as `obj.__attr` fails from outside... yet the mangled name remains perfectly readable. Its true purpose is avoiding accidental attribute clashes between parent and child classes — not secrecy.

> 🔍 **Under the Hood:** Mangling is a pure text rewrite performed at compile time, applied only inside class bodies: any `self.__pin` becomes `self._Vault__pin`. Nothing is hidden in memory — `vars(obj)` happily shows the mangled key. "Privacy by agreement," with a rename to make accidental collisions unlikely.

**Syntax:**

```python
class Vault:
    def __init__(self):
        self.__pin = 1234          # stored as _Vault__pin
```

**Example:** watch the rename happen.

In [4]:
class Vault:
    def __init__(self, pin):
        self.__pin = pin            # double underscore -> mangling


v = Vault(4321)

try:
    print(v.__pin)
except AttributeError as err:
    print("AttributeError:", err)

print(v._Vault__pin)                # mangled name still reachable -- clearly rude though
print(vars(v))                      # the instance dict shows the REAL stored name

AttributeError: 'Vault' object has no attribute '__pin'
4321
{'_Vault__pin': 4321}


In [5]:
# Mangling's REAL job: parents and children can both 'safely' use __names
class BaseForm:
    def __init__(self):
        self.__token = "base-secret"       # becomes _BaseForm__token


class ChildForm(BaseForm):
    def __init__(self):
        super().__init__()
        self.__token = "child-secret"      # becomes _ChildForm__token -- NO clash


form = ChildForm()
print(form._BaseForm__token)               # parent kept its own copy
print(form._ChildForm__token)              # child wrote a separate one

base-secret
child-secret


## 5. `@property`: Getters and Setters, Python-style

Other languages write explicit `get_balance()` / `set_balance()` methods; Python hides validation behind what still *looks* like a plain attribute. `@property` exposes a getter; `@name.setter` attaches validation to assignment. Callers keep clean `obj.balance` syntax while the object keeps control — validate once, and the invariant holds forever.

**Syntax:**

```python
class C:
    @property
    def value(self):                  # getter: obj.value
        return self._value

    @value.setter
    def value(self, new):             # setter: obj.value = x
        if bad(new): raise ValueError(...)
        self._value = new             # NOTE: stored in _value, NOT self.value!
```

**Example:** type-checked, range-checked balance.

In [6]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance            # routes through the SETTER below

    @property
    def balance(self):                    # GETTER: acc.balance
        return self._balance

    @balance.setter
    def balance(self, value):             # SETTER: acc.balance = x
        if not isinstance(value, (int, float)):
            raise TypeError("balance must be a number")
        if value < 0:
            raise ValueError("balance cannot be negative")
        self._balance = value


acc = Account("Sarah", 100)
print(acc.balance)                        # looks like an attribute, runs the getter

acc.balance = 250                         # looks like assignment, runs the setter
print(acc.balance)

for bad in (-5, "lots"):
    try:
        acc.balance = bad
    except (TypeError, ValueError) as err:
        print(f"{err.__class__.__name__}: {err}")

100
250
ValueError: balance cannot be negative
TypeError: balance must be a number


In [7]:
# Validation also guards behavior methods, so EVERY path stays honest:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("balance cannot be negative")
        self._balance = value

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount            # goes through the validating setter

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("insufficient funds")
        self.balance -= amount


acc = Account("Rafi", 200)
acc.deposit(300)
print(acc.balance)

try:
    acc.deposit(-50)                      # sneaky deposit blocked
except ValueError as err:
    print("ValueError:", err)
print(acc.balance)

500
ValueError: deposit must be positive
500


## 6. Read-only and Computed Properties

A property with **no setter** is read-only: assigning raises `AttributeError`. Because a property is just code wearing an attribute's clothes, it can also compute answers on demand — derived values stay in sync automatically.

**Syntax:**

```python
@property
def monthly_pay(self):          # getter only => read-only + computed
    return round(self.salary / 12, 2)
```

**Example:**

In [8]:
class Employee:
    def __init__(self, name, annual_salary):
        self.name = name
        self.annual_salary = annual_salary

    @property
    def monthly_pay(self):                # getter only => read-only, computed
        return round(self.annual_salary / 12, 2)


e = Employee("Amina", 720000)
print(e.monthly_pay)

try:
    e.monthly_pay = 10_000                # no setter exists
except AttributeError as err:
    print("AttributeError:", err)

60000.0
AttributeError: property 'monthly_pay' of 'Employee' object has no setter


In [9]:
# A two-way computed property: set EITHER unit, both stay consistent
class Thermometer:
    def __init__(self, celsius=0.0):
        self.celsius = celsius

    @property
    def fahrenheit(self):
        return round(self.celsius * 9 / 5 + 32, 1)

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = round((value - 32) * 5 / 9, 1)


t = Thermometer(37.0)
print(t.fahrenheit)
t.fahrenheit = 212.0                       # setting the DERIVED value...
print(t.celsius)                           # ...updates the source of truth

98.6
100.0


## 7. Completing the Bank: Invariants Enforced

Our `BankAccount` from Lessons 1–2 now grows up: validated balance, internal ledger behind underscores, and a statement method. Every route into the money passes through exactly one guarded gate — which is the whole point of encapsulation.

**Example:** the finished article.

In [10]:
class BankAccount:
    """Validated state, guarded internals, audit trail."""

    BANK_CODE = "PNB-08"                              # class constant

    def __init__(self, owner, opening_balance=0):
        self.owner = owner                            # public on purpose
        self._transactions = []                       # internal ledger
        self.balance = opening_balance                # via validating setter

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError("balance must be numeric")
        if value < 0:
            raise ValueError("balance cannot go negative")
        self._balance = value

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount
        self._log("deposit", amount)

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError(f"cannot withdraw {amount}; only {self.balance} available")
        self.balance -= amount
        self._log("withdraw", amount)

    def _log(self, kind, amount):                     # internal helper
        self._transactions.append((kind, amount))

    def statement(self):
        lines = [f"Statement {self.BANK_CODE} | {self.owner}"]
        for kind, amount in self._transactions:
            sign = "+" if kind == "deposit" else "-"
            lines.append(f"  {kind:<8} {sign}{amount}")
        lines.append(f"  closing balance: {self.balance}")
        return "\n".join(lines)


acc = BankAccount("Sarah", 1000)
acc.deposit(2500)
acc.withdraw(700)
acc.deposit(120)
print(acc.statement())

for attempt in (10_000, -3):
    try:
        acc.withdraw(attempt)
    except ValueError as err:
        print("blocked:", err)
print("final balance:", acc.balance)      # invariant intact: never negative

Statement PNB-08 | Sarah
  deposit  +2500
  withdraw -700
  deposit  +120
  closing balance: 2920
blocked: cannot withdraw 10000; only 2920 available
final balance: 2923


In [11]:
outside = BankAccount("Auditor", 500)

# The protected route fails loudly:
try:
    outside.balance = -1
except ValueError as err:
    print("setter blocked it:", err)

# The rude route stays physically possible -- privacy by agreement:
outside._transactions.clear()
print("ledger entries after meddling:", outside._transactions)
print("money math still correct:", outside.balance)

setter blocked it: balance cannot go negative
ledger entries after meddling: []
money math still correct: 500


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Trusting `__attr` as security | Mangling is just a rename (`obj._Cls__attr`); nothing is truly private | Treat `__` as clash-avoidance + intent; put real security outside Python |
| Writing `self.value = value` inside its own setter | Infinite recursion → `RecursionError` | Store into the underscore slot: `self._value = value` |
| Heavy computation in a property | Looks as cheap as `.name`, secretly scans files or crunches numbers | Promote expensive work to a method: `compute_statement()` |
| Handing out internal mutable lists/dicts | Callers mutate your internals without passing validation | Return copies or tuples, or expose append-style methods |
| Prefixing everything with `__` "just in case" | Ugly tracebacks, subclass headaches, no extra safety | Default to `_name`; reserve `__name` for subclass-clash avoidance |

## 💡 Best Practices & Pro Tips

- Validate at the boundary: setters plus method guards keep invariants true forever, no matter who calls.
- Keep the public API small and the internals generous — a narrow surface is easy to keep correct.
- Properties for cheap derived values (`monthly_pay`, `fahrenheit`); methods for actions and anything slow.
- 🤖 **AI-engineering relevance:** scikit-learn marks fitted state with a *trailing* underscore — `model.coef_`, `model.classes_`, `model.feature_names_in_` exist only after `.fit()`. Reading those conventions tells you instantly what is internal vs. public in every ML library you touch.
- When using third-party packages, treat their `_underscore` names as off-limits; authors promote names to public deliberately.
- Module takeaway — the full OOP arc: classes (blueprints) → objects (instances with state) → inheritance (specialization) → polymorphism (interchangeable behavior) → encapsulation (guaranteed invariants).

## 📌 Summary

| Convention / Tool | Signal / Behavior | Example |
|---|---|---|
| `name` | Public API — fair game | `acc.owner` |
| `_name` | Internal; handle with care | `acc._transactions` |
| `__name` | Name-mangled to `_Class__name` | `self.__pin` → `_Vault__pin` |
| `@property` | Method disguised as an attribute (getter) | `acc.balance` |
| `@x.setter` | Validates assignment | `acc.balance = -1` → `ValueError` |
| getter-only property | Read-only, possibly computed | `e.monthly_pay` |

**Key takeaways**
- Python enforces nothing — encapsulation is convention plus design discipline ("consenting adults").
- `_name` says *hands off*; `__name` renames itself mainly to survive subclassing.
- `@property` gives Java-grade control with plain-attribute syntax: validate in setters, store in `_slots`.
- Encapsulation earns its keep wherever invariants exist — a balance that can never quietly go negative.

> 🔗 **Next Lesson:** Module 08 complete! Head to [Module 10 · NumPy](../10_NumPy/) — every array you meet there is an object, and now you know precisely what that means. (Module 09 lands soon.)